In [ ]:
!pip install supervision ultralytics

In [ ]:
import cv2
from ultralytics import YOLO
import supervision as sv

image = cv2.imread("park2.png")
model = YOLO("yolov8n.pt")


In [ ]:
results = model(image)[0]
detections = sv.Detections.from_ultralytics(results)
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

In [ ]:
labels = [
    f"{results.names[c]} {c}"
    for c, conf in zip(detections.class_id, detections.confidence)
]
annotated = box_annotator.annotate(image.copy(), detections = detections)
annotated = label_annotator.annotate(annotated, detections = detections, labels = labels)
sv.plot_image(annotated)

In [ ]:
def showImage(det):
    labels = [
        f"{results.names[c]} {conf:.0%}"
        for c, conf in zip(det.class_id, det.confidence)
    ]
    annotated = box_annotator.annotate(image.copy(), detections = det)
    annotated = label_annotator.annotate(annotated, detections = det, labels = labels)
    sv.plot_image(annotated)

In [ ]:
showImage(detections)

# Detection by Element

In [ ]:
detections_by_element = detections[4]
showImage(detections_by_element)

# Detections By Confidence

In [ ]:
mask_by_confidence = detections.confidence > 0.85
detections_by_confidence = detections[mask_by_confidence]
showImage(detections_by_confidence)

# Detections By Class

In [ ]:
mask_by_class = detections.class_id == 0
detections_by_class = detections[mask_by_class]
showImage(detections_by_class)

# Detections By Set Of Classes

In [ ]:
import numpy as np
selected_classes = [0, 1, 24] # 0 -> person, 1 = bycicle, 24 = backpack
mask_by_set_of_classes = np.isin(detections.class_id, selected_classes)
detections_by_set_of_classes = detections[mask_by_set_of_classes]
showImage(detections_by_set_of_classes)

# Detection By Area

In [ ]:
area_mask = detections.area > 50000
detections_by_mask = detections[area_mask]
showImage(detections_by_mask)

# Detections By Relative Area

In [ ]:
height, width, channels = image.shape
image_area = width * height
mask_by_relative_area = (detections.area / image_area) < 0.01
detections_by_relative_area = detections[mask_by_relative_area]
showImage(detections_by_relative_area)

# Detections By Box Dimensions

In [ ]:
w = detections.xyxy[:,2] - detections.xyxy[:, 0]
h = detections.xyxy[:, 3] - detections.xyxy[:, 1]
mask_by_box_dimensions = (w > 300) & (h > 500)
detections_by_box_dimensions = detections[mask_by_box_dimensions]
showImage(detections_by_box_dimensions)

# Mask By Position

In [ ]:
half_x = (detections.xyxy[:, 0] + detections.xyxy[:, 2]) // 2
half_y = (detections.xyxy[:, 1] + detections.xyxy[:, 3]) // 2
mask_by_position = (half_x < width // 2) & (half_y > height // 2)
detections_by_position = detections[mask_by_position]
showImage(detections_by_position)

## NMS — Non-Maximum Suppression

A veces el modelo detecta el mismo objeto múltiples veces con cajas ligeramente diferentes.
NMS conserva solo la caja de mayor confianza cuando hay demasiado solapamiento.

In [ ]:
results_baja = model(image, conf=0.3)[0]
results_alta = model(image, conf=0.7)[0]
det_baja = sv.Detections.from_ultralytics(results_baja)
det_alta = sv.Detections.from_ultralytics(results_alta)

mezclado = sv.Detections.merge([det_baja, det_alta])
print(f"Detecciones individuales: baja_conf={len(det_baja)}, alta_conf={len(det_alta)}")
print(f"Después de merge (duplicados incluidos): {len(mezclado)}")

# threshold=0.5 → si dos cajas se solapan más del 50%, NMS elimina la de menor confianza
sin_duplicados = mezclado.with_nms(threshold=0.5)
print(f"Después de NMS (sin duplicados): {len(sin_duplicados)}")

showImage(mezclado)
showImage(sin_duplicados)